# 01 Data Quality Correction

**1. Standardize columns**
- Invalid column names
- drop unnecessary columns

**2. Normalize representations**
- String whitespace
- Case normalization
- Common categorical variations
- Datatype inconsistencies

**3. Type conversion**
- Numeric strings → numeric
- Date strings → datetime
- Boolean representations → boolean

**4. Invalid-value handling**
- Negative values where impossible
- Invalid categorical values
- Invalid ranges

## Imports

In [1]:
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings("ignore")

## Create/Load Sample Dataset

In [2]:
df = pd.DataFrame(
    {
        "Customer ID": [1, 2, 3, 4, 5, 5, 6, 7, 8, 9, 10, 11],
        " Customer-Name ": [
            " Rahul ", "priya", "AMIT", " Sneha", None, "Rohit", "Neha", "rahul", "Priya ", "Amit", "Sneha", "Rohit",
        ],
        "AGE": [ 25, 31, None, 29, -5, 42, 150, "35", 28, None, 31, 42,
        ],
        "Annual Income ($)": [ 
            50000, 62000, 55000, None, 45000, 72000, 80000, "65000", -10000, 58000, None, 72000, 
            ],
        "City/Location": [
            "Mumbai", "mumbai", " MUMBAI ", "Pune", None, "pune", "Delhi", "Delhi ", "delhi", "Mumbai", "Pune", "Delhi", 
            ],
        "Gender Type": [
            "Male", "female", "M", "Female ", None, "male", "F", "Male", "Female", "unknown", "M", "male",
            ],
        "Signup Date": [
            "2026-01-15", "2026-02-20", "15/03/2026", "2026-04-10", None, "2026-05-12", "invalid-date", "2026-06-20", "20/07/2026", "2026-08-01", "2026-08-10", "2026-08-15",
        ],
        "Is Active?": [
            True, "True", "yes", "No", None, 1, 0, "true", "false", "yes", "Y", "N",
        ],
    }
)

df

,Customer ID,Customer-Name,AGE,Annual Income ($),City/Location,Gender Type,Signup Date,Is Active?
0,1,Rahul,25,50000,Mumbai,Male,2026-01-15,True
1,2,priya,31,62000,mumbai,female,2026-02-20,True
2,3,AMIT,None,55000,MUMBAI,M,15/03/2026,yes
3,4,Sneha,29,None,Pune,Female,2026-04-10,No
4,5,NaN,-5,45000,NaN,NaN,NaN,None
5,5,Rohit,42,72000,pune,male,2026-05-12,1
6,6,Neha,150,80000,Delhi,F,invalid-date,0
7,7,rahul,35,65000,Delhi,Male,2026-06-20,true
8,8,Priya,28,-10000,delhi,Female,20/07/2026,false
9,9,Amit,None,58000,Mumbai,unknown,2026-08-01,yes


In [3]:
from pathlib import Path

data_dir = Path("data")
data_dir.mkdir(parents=True, exist_ok=True)

output_path = data_dir / "data_quality_sample.csv"

df.to_csv(output_path, index=False)

print(f"Saved dataset to: {output_path}")

Saved dataset to: data/data_quality_sample.csv


## 1. Standardize Columns

```
Customer ID       → customer_id
 Customer-Name    → customer_name
AGE               → age
Annual Income ($) → annual_income
City/Location     → city_location
Gender Type       → gender_type
Signup Date       → signup_date
Is Active?        → is_active
```

In [4]:
df.columns

Index(['Customer ID', ' Customer-Name ', 'AGE', 'Annual Income ($)',
       'City/Location', 'Gender Type', 'Signup Date', 'Is Active?'],
      dtype='str')

In [5]:
column_map = {
    'Customer ID': 'customer_id', 
    ' Customer-Name ': 'customer_name', 
    'AGE': 'age', 
    'Annual Income ($)': 'annual_income',
    'City/Location': 'city', 
    'Gender Type': 'gender_type', 
    'Signup Date': 'signup_date', 
    'Is Active?': 'is_active'
}

In [6]:
df = df.rename(columns=column_map)

In [7]:
df.columns

Index(['customer_id', 'customer_name', 'age', 'annual_income', 'city',
       'gender_type', 'signup_date', 'is_active'],
      dtype='str')

## 2. Normalize representations

### 2.1 Numerical

In [8]:
NUM_DF = df[['age', 'annual_income']]
NUM_DF

,age,annual_income
0,25,50000
1,31,62000
2,None,55000
3,29,None
4,-5,45000
5,42,72000
6,150,80000
7,35,65000
8,28,-10000
9,None,58000


#### 2.1.1. None to np.nan and convert values to fload64 

In [9]:
NUM_DF['age'] = pd.to_numeric(df['age'], errors='coerce')
NUM_DF['annual_income'] = pd.to_numeric(df['annual_income'], errors='coerce')

In [10]:
NUM_DF

,age,annual_income
0,25.0,50000.0
1,31.0,62000.0
2,NaN,55000.0
3,29.0,NaN
4,-5.0,45000.0
5,42.0,72000.0
6,150.0,80000.0
7,35.0,65000.0
8,28.0,-10000.0
9,NaN,58000.0


#### 2.1.2. Alternate way: None to np.nan and convert values to int64

In [11]:
# alternate way: None to np.nan and convert values to int64
NUM_DF['age'] = df['age'].replace({None: np.nan})
NUM_DF['annual_income'] = df['annual_income'].replace({None: np.nan})

In [12]:
NUM_DF

,age,annual_income
0,25,50000
1,31,62000
2,NaN,55000
3,29,NaN
4,-5,45000
5,42,72000
6,150,80000
7,35,65000
8,28,-10000
9,NaN,58000


### 2.2 Categorical

In [13]:
CAT_DF = df[['customer_name', 'city', 'gender_type','is_active']]
CAT_DF

,customer_name,city,gender_type,is_active
0,Rahul,Mumbai,Male,True
1,priya,mumbai,female,True
2,AMIT,MUMBAI,M,yes
3,Sneha,Pune,Female,No
4,NaN,NaN,NaN,None
5,Rohit,pune,male,1
6,Neha,Delhi,F,0
7,rahul,Delhi,Male,true
8,Priya,delhi,Female,false
9,Amit,Mumbai,unknown,yes


#### 2.2.1. Normalize & Standard Formatting (Lowercase strings, strip whitespace)

##### 1. Normalizing

In [14]:
string_cols = CAT_DF.select_dtypes(include=['object', 'string']).columns
for col in string_cols:
    CAT_DF[col] = CAT_DF[col].astype(str).str.strip().str.lower()

In [15]:
CAT_DF

,customer_name,city,gender_type,is_active
0,rahul,mumbai,male,true
1,priya,mumbai,female,true
2,amit,mumbai,m,yes
3,sneha,pune,female,no
4,NaN,NaN,NaN,NaN
5,rohit,pune,male,1
6,neha,delhi,f,0
7,rahul,delhi,male,true
8,priya,delhi,female,false
9,amit,mumbai,unknown,yes


##### 2. Standardizing

In [16]:
gender_grouped_map = {
    'male': ['m', 'male'],
    'female': ['f', 'female'],
}

# Flatten into: {'y': 1, 'yes': 1, 'true': 1, ...}
gender_flat_map = {value: target for target, values in gender_grouped_map.items() for value in values}

# Apply mapping
CAT_DF['gender_type'] = CAT_DF['gender_type'].map(gender_flat_map)

In [17]:
is_active_grouped_map = {
    0: [ 0, 'n', 'no', 'false'],
    1: [ 1, 'y', 'yes', 'true']
}

# Flatten into: {'y': 1, 'yes': 1, 'true': 1, ...}
is_active_flat_map = {value: target for target, values in is_active_grouped_map.items() for value in values}

# Apply mapping
CAT_DF['is_active'] = CAT_DF['is_active'].map(is_active_flat_map)

In [18]:
CAT_DF

,customer_name,city,gender_type,is_active
0,rahul,mumbai,male,1.0
1,priya,mumbai,female,1.0
2,amit,mumbai,male,1.0
3,sneha,pune,female,0.0
4,NaN,NaN,NaN,NaN
5,rohit,pune,male,NaN
6,neha,delhi,female,NaN
7,rahul,delhi,male,1.0
8,priya,delhi,female,0.0
9,amit,mumbai,NaN,1.0


#### 2.2.2. Replace None with `missing`

In [19]:
CAT_DF.replace(['None', 'none', 'N/A', 'n/a', '', 'null', 'unknown', np.nan], 'missing')

,customer_name,city,gender_type,is_active
0,rahul,mumbai,male,1.0
1,priya,mumbai,female,1.0
2,amit,mumbai,male,1.0
3,sneha,pune,female,0.0
4,missing,missing,missing,missing
5,rohit,pune,male,missing
6,neha,delhi,female,missing
7,rahul,delhi,male,1.0
8,priya,delhi,female,0.0
9,amit,mumbai,missing,1.0


#### 2.2.3. Replace None with `np.nan`

In [20]:
# Convert string representation variants to actual np.nan
CAT_DF.replace(['None', 'none', 'N/A', 'n/a', '', 'null', 'unknown'], np.nan)

,customer_name,city,gender_type,is_active
0,rahul,mumbai,male,1.0
1,priya,mumbai,female,1.0
2,amit,mumbai,male,1.0
3,sneha,pune,female,0.0
4,NaN,NaN,NaN,NaN
5,rohit,pune,male,NaN
6,neha,delhi,female,NaN
7,rahul,delhi,male,1.0
8,priya,delhi,female,0.0
9,amit,mumbai,NaN,1.0


### 2.3 Date-Time

In [21]:
TIME_DF = df[['signup_date']]
TIME_DF

,signup_date
0,2026-01-15
1,2026-02-20
2,15/03/2026
3,2026-04-10
4,NaN
5,2026-05-12
6,invalid-date
7,2026-06-20
8,20/07/2026
9,2026-08-01


#### 2.3.1 Clean invalid tokens to `np.nan`

In [22]:
TIME_DF['signup_date'] = TIME_DF['signup_date'].replace(['invalid-date', 'NaN', 'null', ''], np.nan)

In [23]:
TIME_DF

,signup_date
0,2026-01-15
1,2026-02-20
2,15/03/2026
3,2026-04-10
4,NaN
5,2026-05-12
6,NaN
7,2026-06-20
8,20/07/2026
9,2026-08-01


#### 2.3.2 Parse mixed string formats to standard datetime objects

In [24]:
TIME_DF['signup_date'] = pd.to_datetime(
    TIME_DF['signup_date'], 
    errors='coerce', 
    format='mixed', 
    dayfirst=True
)

In [25]:
TIME_DF

,signup_date
0,2026-01-15
1,2026-02-20
2,2026-03-15
3,2026-10-04
4,NaT
5,2026-12-05
6,NaT
7,2026-06-20
8,2026-07-20
9,2026-01-08


#### 2.3.3 Format as unified target string: YYYY-MM-DD

In [26]:
# TIME_DF['signup_date'] = TIME_DF['signup_date'].dt.strftime('%Y-%m-%d')
# TIME_DF

In [27]:
TIME_DF.info()

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 1 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   signup_date  10 non-null     datetime64[us]
dtypes: datetime64[us](1)
memory usage: 228.0 bytes


In [28]:
df = pd.concat([
    df[['customer_id']], NUM_DF, CAT_DF, TIME_DF
], axis=1)